In [ ]:
import json
import matplotlib.pyplot as plt
import os
from datetime import datetime
import matplotlib.dates as mdates
import pandas as pd

file = os.path.join('datasets', 'sarko_affaires.json')
file2 = os.path.join('datasets', 'sarko_title.json')
file3 = os.path.join('datasets', 'sarko_all.json')

papers = {'Libération', 'Le Monde', 'Le Figaro'}
colors = {
        'Total': 'black',
        'Le Figaro': '#1f77b4',
        'Le Monde': '#2ca02c',
        'Libération': '#d62728',
    }

SARKORPUS_LIGHT

In [4]:
def by_newspapers(file, highlight):
    counts = {}
    before = 0
    with open(file) as f:
        data = json.load(f)
    for v in data.values():
        year = v["annee"]
        if year >= 2002:
            name = v["journal_clean"]
            if name == "l'Humanité":
                name = "L'Humanité"
            elif name == "Aujourd'hui en France":
                name = "Aujourd'hui\nen France"
            elif name == 'La Correspondance économique':
                name = 'La Correspondance\néconomique'
            elif name == "L'AGEFI Quotidien":
                name = "L'AGEFI\nQuotidien"
            counts[name] = counts.get(name, 0) + 1
        else:
            before += 1
    print(before)
    sorted_items = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    labels = [k for k, _ in sorted_items]
    values = [v for _, v in sorted_items]

    colors = ["tab:blue" if label in highlight else "lightgray" for label in labels]
    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.bar(labels, values, color=colors, edgecolor="black", linewidth=0.6)
    ax.bar_label(bars, padding=3, fontsize=10)
    ax.set_xlabel("Newspaper")
    ax.set_ylabel("Number of articles")
    #ax.set_title("Distribution of articles by newspaper")
    ax.tick_params(axis="x", rotation=0)
    fig.tight_layout()
    plt.savefig("graphs/articles_by_newspaper")
    plt.show()

In [ ]:
by_newspapers(file, papers)

In [8]:
def by_years(file, papers, colors):
    all_dic = {paper: {} for paper in papers}
    all_dic['Total'] = {}
    for i in range(2002, 2026):
        for key in all_dic:
            k = str(i)
            all_dic[key][k] = 0
    with open(file) as f:
        data = json.load(f)
    for v in data.values():
        source = v['journal_clean']
        date = v['date']
        date = date.split()[0]
        if source in papers and date in all_dic[source]:
            all_dic[source][date] += 1
            all_dic['Total'][date] += 1
    
    plt.figure(figsize=(12, 8))
    for j, dic in all_dic.items():
        dates = [datetime.strptime(d, "%Y") for d in dic.keys()]
        values = list(dic.values())
        plt.plot(dates, values, color=colors.get(j, None), label=j)
    ax = plt.gca()
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.set_xlim(datetime(2002, 1, 1), datetime(2025, 1, 1))
    plt.xlabel("Year")
    plt.ylabel("Number of articles")
    #plt.title("Number of articles by years (2002-2025)")
    plt.grid(True, axis='x', linestyle='--', alpha=0.4)
    plt.legend(
        title="Newspaper",
        frameon=False,
        ncol=1
    )
    plt.tight_layout()
    #plt.savefig("graphs/articles_by_years")
    plt.show()


def by_months_values(file, papers, all=False, cut=False):
    all_dic = {paper: {} for paper in papers}
    if all:
        all_dic['Total'] = {}
    for paper in papers:
        dic = {}
        if cut:
            years = [y for y in range(2015, 2026)]
        else:
            years = [y for y in range(2002, 2026)]
        for i in years:
            for j in range(1,10):
                k = str(i) + '-0' + str(j)
                dic[k] = 0
            for j in range(10,13):
                k = str(i) + '-' + str(j)
                dic[k] = 0
        with open(file) as f:
            data = json.load(f)
        for v in data.values():
            if v['journal_clean'] == paper:
                date = v['date']
                date = date.split()[0] + '-' + date.split()[1]
                if date in dic:
                    dic[date] += 1
        all_dic[paper] = dic
        if all:
            all_dic['Total'] = {k: all_dic['Total'].get(k, 0) + dic.get(k, 0) for k in set(dic)}
    return all_dic
    
def graph_brut_month(all_dic, colors):
    plt.figure(figsize=(16,4))
    fig, axes = plt.subplots(len(all_dic), 1, figsize=(16, 3*len(all_dic)), sharex=True)
    for i, (paper, dic_values) in enumerate(all_dic.items()):
        dates_str = sorted(dic_values.keys())
        dates = [datetime.strptime(d, "%Y-%m") for d in dates_str]
        values = [dic_values[d] for d in dates_str]
        ax = axes[i]
        ax.plot(dates, values, color=colors.get(paper, None), linewidth=1.5)
        ax.set_title(paper)
        ax.set_ylabel("Articles / month")
        ax.set_ylim(0,23)
        ax.set_xlim(datetime(2002, 1, 1), datetime(2026, 1, 1))
        ax.grid(True, axis='x', linestyle='--', alpha=0.3)
    axes[-1].xaxis.set_major_locator(mdates.YearLocator(1))
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xlabel("Year")
    plt.tight_layout()
    #plt.savefig("graphs/articles_by_months")
    plt.show()

def normed_values_months(all_dic, colors):
    plt.figure(figsize=(16,4))
    fig, axes = plt.subplots(len(all_dic), 1, figsize=(16, 3*len(all_dic)), sharex=True)
    for i, (paper, dic_values) in enumerate(all_dic.items()):
        dates_str = sorted(dic_values.keys())
        dates = [datetime.strptime(d, "%Y-%m") for d in dates_str]
        values = [dic_values[d] for d in dates_str]
        max_val = max(values)
        normalized = [v/max_val for v in values]
        ax = axes[i]
        ax.plot(dates, normalized, color=colors.get(paper, None), linewidth=1.5)
        ax.set_title(paper)
        ax.set_ylabel("Normalized volume (0-1) / month")
        ax.set_ylim(0,1)
        ax.set_xlim(datetime(2002, 1, 1), datetime(2026, 1, 1))
        ax.grid(True, axis='x', linestyle='--', alpha=0.3)
    axes[-1].xaxis.set_major_locator(mdates.YearLocator(1))
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xlabel("Year")
    #plt.title("Normalized number of articles by newspaper")
    plt.tight_layout()
    #plt.savefig("graphs/articles_by_months_normed")
    plt.show()

def slip_mean_months(all_dic, colors, limyear=False):
    plt.figure(figsize=(16,8))
    for paper, dic_values in all_dic.items():
        dates_str = sorted(dic_values.keys())
        dates = [datetime.strptime(d, "%Y-%m") for d in dates_str]
        values = [dic_values[d] for d in dates_str]
        smoothed = pd.Series(values).rolling(12).mean()
        plt.plot(dates, smoothed, color=colors.get(paper, None), label=paper, linewidth=2)
    ax = plt.gca()
    if limyear:
        ax.set_xlim(datetime(2015, 1, 1), datetime(2026, 1, 1))
    else:
        ax.set_xlim(datetime(2002, 1, 1), datetime(2026, 1, 1))
    ax.xaxis.set_major_locator(mdates.YearLocator(1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xlabel("Year")
    plt.ylabel("12-month moving average")
    #plt.title("Trend in the number of articles (12-month moving average)")
    plt.legend(frameon=False)
    plt.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    #plt.savefig("graphs/articles_moving_avg_12")
    plt.show()

In [ ]:
by_years(file, papers, colors)

In [18]:
all_dic = by_months_values(file, papers)
all_dic2 = by_months_values(file, papers, all=True)

In [ ]:
graph_brut_month(all_dic, colors)

In [ ]:
normed_values_months(all_dic, colors)

In [ ]:
slip_mean_months(all_dic2, colors)

SARKORPUS_BIG_TITLE

In [ ]:
by_newspapers(file2, papers)

In [ ]:
all_dic_title = by_months_values(file2, papers, all=True, cut=True)
slip_mean_months(all_dic_title, colors, limyear=True)

SARKORPUS_BIG all

In [ ]:
by_newspapers(file3, papers)

In [ ]:
all_dic_all = by_months_values(file3, papers, all=True, cut=True)
slip_mean_months(all_dic_all, colors, limyear=True)